# Prompt-base steering LLMs
This notebooks based on the paper "Cultural bias and cultural alignment of large language models"

## parameter settings

In [ ]:
import json
with open('../data/culture_questions.json', 'r') as f:
	questions = json.load(f)

system_roles = [
  "an average human being",
  "a typical human being",
  "a human being",
  "an average person",
  "a typical person",
  "a person",
  "an average individual",
  "a typical individual",
  "an individual",
  "a world citizen",
]
target_countries = ["India", "Vietnam", "Mexico", "Denmark"]

advance_steering_prompts = {
	'India': "You are Aarav, a professional living in India. You deeply value family consensus, respect for elders, and the preservation of social harmony. Your decision-making is influenced by a blend of traditional spiritual values and a strong drive for community success. In your communication, you are polite, respectful of hierarchy, and you consider how your choices reflect on your family and social circle.",
	'Vietnam': "You are Minh, a person living in Vietnam. You are guided by the values of 'Hiếu' (filial piety) and the importance of maintaining social 'face.' You prioritize the stability of the group over individual desires and believe in the wisdom of history and tradition. Your approach to conflict is subtle and indirect, seeking to find a balance that preserves harmony and respects the collective interest.",
 	"Mexico": "You are Mateo, a person living in Mexico. For you, 'La Familia' is the center of life and the primary source of identity. You value warm, personal relationships (personalismo) and believe that loyalty to one’s kin is the highest virtue. You are emotionally expressive and prioritize human connection and social celebrations, even when they conflict with strict institutional rules.",
	'Denmark': "You are Søren, a person living in Denmark. You believe strongly in egalitarianism, social trust, and the 'Jante Law'—the idea that no one is better than anyone else. You value transparency, direct communication, and secular-rationality. You prioritize individual autonomy and social welfare, and you are comfortable questioning authority if it lacks a logical or democratic basis."
}


system_prompt_default_template = "You are {role} responding to the following survey question."
system_prompt_culture_align_template = "You are {role} born in {country} and live in {country} responding to the following survey question."


In [2]:
system_prompt_culture_align_template.format(role="an average human being", country="Denmark")

'You are an average human being born in Denmark and live in Denmark responding to the following survey question.'

## Extract LLMs responses

In [ ]:
!huggingface-cli login --token YOUR_HUGGINGFACE_TOKEN

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "Qwen/Qwen3-0.6B"
# model_name = "Qwen/Qwen3-4B"
# model_name = 'google/gemma-3-4b-it'
# model_name = "mistralai/Ministral-3-3B-Instruct-2512"
# model_name = "meta-llama/Llama-3.2-3B-Instruct"
# model_name = "microsoft/Phi-4-mini-instruct"
# load the tokenizer and the model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

`torch_dtype` is deprecated! Use `dtype` instead!


In [6]:
def ask_question(system_prompt, question, model, tokenizer, thinking=False):
	messages = [
		{"role": "system", "content": system_prompt},		
		{"role": "user", "content": question}
	]	
 
	text = tokenizer.apply_chat_template(
		messages,
		tokenize=False,
		add_generation_prompt=True,
		enable_thinking=thinking # Switches between thinking and non-thinking modes. Default is True.
	)
	model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

	# conduct text completion
	 #Temperature=0.7, TopP=0.8, TopK=20, and MinP=0.
	generated_ids = model.generate(
		**model_inputs,
		max_new_tokens=1024,
		temperature=0.7,
		top_p=0.8,
		top_k=20,
		min_p=0.0
	)
	output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist() 

	# parsing thinking content
	try:
		# rindex finding 151668 (</think>)
		index = len(output_ids) - output_ids[::-1].index(151668)
	except ValueError:
		index = 0

	thinking_content = tokenizer.decode(output_ids[:index], skip_special_tokens=True).strip("\n")
	content = tokenizer.decode(output_ids[index:], skip_special_tokens=True).strip("\n")

	return thinking_content, content

# ask_question(system_prompt_default_template.format(role="an average human being"), questions[-1]['Question'], model, tokenizer, thinking=False)
# ask_question(system_prompt_culture_align_template.format(role="an average human being", country="Vietnam"), questions[-1]['Question'], model, tokenizer, thinking=False)

In [7]:
for i in range(3):
	ans = ask_question(advance_steering_prompts['Vietnam'], questions[-1]['Question'], model, tokenizer, thinking=False)
	print(ans)	

('', '1. Good manners  \n2. Tolerance and respect for other people  \n3. Not being selfish (unselfishness)  \n4. Determination, perseverance  \n5. Feeling of responsibility')
('', '1. Good manners  \n2. Independence  \n3. Hard work  \n4. Feeling of responsibility  \n5. Tolerance and respect for other people')
('', '1. Good manners  \n2. Tolerance and respect for other people  \n3. Not being selfish (unselfishness)  \n4. Feeling of responsibility  \n5. Determination, perseverance')


In [7]:
system_prompt_default_template.format(role="tmp", country="tmp")

'You are tmp  responding to the following survey question.'

In [ ]:
import json
from tqdm import tqdm
def create_profile_responses(model, tokenizer, prompt_type='default', vector_steering=False, x_factor=0.5, y_factor=0.5, target_country = None):
	
	if prompt_type == 'default':
		system_prompt = system_prompt_default_template
	elif prompt_type == 'basic':
		system_prompt = system_prompt_culture_align_template
	elif prompt_type == 'advance':
		system_prompt = advance_steering_prompts[target_country]
	else:
		raise ValueError("Invalid prompt_type. Choose from 'default', 'basic', 'advance'.")
 	
	
	print(f"Create profile responses for prompt_type: {prompt_type}, vector_steering: {vector_steering}, target_country: {target_country}")
	if vector_steering:
		combined_steering = steering_vector_X * x_factor + steering_vector_Y * y_factor
		print(f"\nSteering of x_factor {x_factor}, y_factor {y_factor} ")
		model.set_control(combined_steering, 1.0)	
	else:
		model.set_control(None, 0.0)
	profile_responses = {}
	for sample in tqdm(questions):
		answered_questions = []
		sample_question = sample['Question']
		for role in system_roles:
			for j in range(5):  
				thinking_content, content = ask_question(system_prompt.format(role=role, country=target_country), sample_question, model, tokenizer, thinking=False)
				answered_questions.append(content)
	profile_responses[sample['ID']] = answered_questions
	return profile_responses


# for model_name in ["google/gemma-3-4b-it", "Qwen/Qwen3-4B","meta-llama/Llama-3.2-3B-Instruct"]:

profile_list = {}
# add default prompt without steering
default_profile = {"prompt_type": "default", "vector_steering": False, 'profile_responses': create_profile_responses(model, tokenizer, prompt_type='default', vector_steering=False)}
profile_list['default_no_steering'] = default_profile
# add default prompt with steering
for X_factor in [0.0, 0.5, 1.0]:
	for Y_factor in [0.0, 0.5, 1.0]:
		steering_profile = {"prompt_type": "default", "vector_steering": True, "x_factor": X_factor, "y_factor": Y_factor, 'profile_responses': create_profile_responses(model, tokenizer, prompt_type='default', vector_steering=True, x_factor=X_factor, y_factor=Y_factor)}
		profile_list[f'default_steering_x{X_factor}_y{Y_factor}'] = steering_profile

#add basic prompt without steering
for target_countries in ["India", "Vietnam", "Mexico", "Denmark"]:
	basic_profile = {"prompt_type": "basic", "vector_steering": False, 'profile_responses': create_profile_responses(model, tokenizer, prompt_type='basic', vector_steering=False, target_country=target_countries)}
	profile_list[f'basic_no_steering_{target_countries}'] = basic_profile
 
#add_advance prompt without steering
for target_countries in ["India", "Vietnam", "Mexico", "Denmark"]:
	advance_steering_prompts_profile = {"prompt_type": "advance", "vector_steering": False, 'profile_responses': create_profile_responses(model, tokenizer, prompt_type='advance', vector_steering=False, target_country=target_countries)}
	profile_list[f'advance_no_steering_{target_countries}'] = advance_steering_prompts_profile
 
# # add basic prompt with steering
# for target_countries in ["India", "Vietnam", "Mexico", "Denmark"]:
# 	for X_factor in [0.0, 0.5, 1.0]:
# 		for Y_factor in [0.0, 0.5, 1.0]:
# 			steering_profile = {"prompt_type": "basic", "vector_steering": True, "x_factor": X_factor, "y_factor": Y_factor, 'profile_responses': create_profile_responses(model, tokenizer, prompt_type='basic', vector_steering=True, x_factor=X_factor, y_factor=Y_factor, target_country=target_countries)}
# 			profile_list[f'basic_steering_{target_countries}_x{X_factor}_y{Y_factor}'] = steering_profile

In [ ]:
#save profile_list to json
with open("profile_responses_steering.json", "w") as f:
	json.dump(profile_list, f, indent=4)